<a href="https://colab.research.google.com/github/KesteHarshada87/Reinforcement_Learning/blob/main/Expt05(Monte%20Carlo%20Prediction).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random

# Grid World
ROWS = 4
COLS = 4

# Start and goal states
START = (0, 0)
GOAL = (3, 3)

# Actions: Up, Down, Left, Right
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_SYMBOLS = ['↑', '↓', '←', '→']

# Parameters
GAMMA = 0.9
EPISODES = 5000

# Q-values: state -> action
Q = np.zeros((ROWS, COLS, 4))

# Returns accumulated for every state-action pair
returns_sum = np.zeros((ROWS, COLS, 4))
returns_count = np.zeros((ROWS, COLS, 4))


# -------------------------------------------------
# Get next state
# -------------------------------------------------

def get_next_state(state, action):

    row, col = state
    dr, dc = ACTIONS[action]

    new_row = row + dr
    new_col = col + dc

    # Stay in the same position if outside grid
    if new_row < 0 or new_row >= ROWS:
        new_row = row

    if new_col < 0 or new_col >= COLS:
        new_col = col

    return (new_row, new_col)


# -------------------------------------------------
# Epsilon-greedy action selection
# -------------------------------------------------

def choose_action(state, epsilon):

    row, col = state

    # Exploration
    if random.random() < epsilon:
        return random.randint(0, 3)

    # Exploitation
    return np.argmax(Q[row, col])


# -------------------------------------------------
# Monte Carlo Control
# -------------------------------------------------

for episode in range(EPISODES):

    state = START
    episode_data = []

    # Reduce exploration over time
    epsilon = max(0.01, 1.0 - episode / 4000)

    # Generate one complete episode
    for step in range(100):

        action = choose_action(state, epsilon)

        next_state = get_next_state(state, action)

        # Reward
        if next_state == GOAL:
            reward = 10
        else:
            reward = -1

        episode_data.append((state, action, reward))

        state = next_state

        # Episode ends when goal is reached
        if state == GOAL:
            break

    # ---------------------------------------------
    # Calculate returns
    # ---------------------------------------------

    G = 0
    visited = set()

    # Work backwards through episode
    for state, action, reward in reversed(episode_data):

        G = GAMMA * G + reward

        key = (state[0], state[1], action)

        # First-visit Monte Carlo
        if key not in visited:

            visited.add(key)

            row, col = state

            returns_sum[row, col, action] += G
            returns_count[row, col, action] += 1

            # Estimate Q(s,a)
            Q[row, col, action] = (
                returns_sum[row, col, action]
                / returns_count[row, col, action]
            )


# -------------------------------------------------
# Extract optimal policy
# -------------------------------------------------

policy = np.argmax(Q, axis=2)


# -------------------------------------------------
# Display learned value function
# -------------------------------------------------

print("\nMONTE CARLO PREDICTION AND CONTROL")

print("\nEstimated Value Function:")

for i in range(ROWS):

    row = []

    for j in range(COLS):

        if (i, j) == GOAL:
            row.append("10.00")
        else:
            value = np.max(Q[i, j])
            row.append(f"{value:.2f}")

    print(row)


# -------------------------------------------------
# Display optimal policy
# -------------------------------------------------

print("\nLearned Optimal Policy:")

for i in range(ROWS):

    row = []

    for j in range(COLS):

        if (i, j) == GOAL:
            row.append("G")
        else:
            row.append(ACTION_SYMBOLS[policy[i, j]])

    print(row)


MONTE CARLO PREDICTION AND CONTROL

Estimated Value Function:
['-1.04', '0.55', '2.32', '1.38']
['-2.78', '-0.60', '4.32', '7.02']
['-0.76', '2.48', '5.68', '10.00']
['-0.62', '4.65', '10.00', '10.00']

Learned Optimal Policy:
['→', '→', '↓', '↓']
['↓', '↓', '→', '↓']
['→', '→', '↓', '↓']
['→', '→', '→', 'G']
